In [6]:
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

In [7]:
x_train = pd.read_csv('x_train_scaled.csv')
x_test = pd.read_csv('x_test_scaled.csv')
y_train = pd.read_csv('y_train.csv')
y_test = pd.read_csv('y_test.csv')

x_train.head()

,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,-0.022803,-1.576975,-0.100746,-0.180840,-0.151997,-0.672522,-0.063764,-0.900750,-0.057706,-0.666180
1,-0.020393,0.184209,-0.100746,-0.180416,-0.334990,0.493412,-0.063764,-0.017611,-0.057706,-0.666180
2,-0.022803,-0.628645,-0.100746,-0.180824,-0.164834,-1.255488,-0.063764,-0.900750,-0.057706,2.944145
3,-0.021217,-0.831859,-0.100746,-0.180815,0.187732,-1.255488,-0.063764,-0.900750,-0.057706,3.846727
4,-0.022803,-1.170548,-0.100746,-0.180777,-0.045659,0.687734,-0.063764,-0.017611,-0.057706,-0.666180


In [8]:
## Train a baseline logistic regression model_1
model_1 = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=1000,
    solver="lbfgs",
    n_jobs=-1,
    class_weight="balanced"
)

model_1.fit(x_train, y_train)

C:\Users\Saad\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# predicted probabilities for the positive class (1)
y_proba = model_1.predict_proba(x_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)  # can tune this threshold

auc = roc_auc_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred)

print("AUC-ROC:", auc)
print("Confusion matrix:\n", cm)
print("\nClassification report:\n", classification_report(y_test, y_pred))


AUC-ROC: 0.7899659904870495
Confusion matrix:
 [[21949  6095]
 [  688  1268]]

Classification report:
               precision    recall  f1-score   support

           0       0.97      0.78      0.87     28044
           1       0.17      0.65      0.27      1956

    accuracy                           0.77     30000
   macro avg       0.57      0.72      0.57     30000
weighted avg       0.92      0.77      0.83     30000



In [10]:
test_data = pd.read_csv('cs-test.csv')
test_ids = test_data['Unnamed: 0']
test_data_processed = pd.read_csv('processed_test_data.csv')

test_proba = model_1.predict_proba(test_data_processed)[:, 1]

test_data_processed.head()



,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,-0.019219,-0.628645,-0.100746,-0.180755,-0.052952,-0.866844,-0.063764,-0.900750,-0.057706,-0.666180
1,-0.020928,0.319685,-0.100746,-0.180574,0.198016,1.270700,-0.063764,2.631804,-0.057706,1.138982
2,-0.022628,0.455161,-0.100746,-0.180492,-0.097953,0.687734,-0.063764,-0.017611,-0.057706,1.138982
3,-0.021669,-0.967335,0.137824,-0.180369,-0.235288,-0.283877,-0.063764,0.865527,-0.057706,-0.666180
4,-0.018755,-1.712451,-0.100746,-0.180836,-0.186787,-0.866844,-0.063764,-0.900750,-0.057706,0.236401


In [11]:
### Gradient Booster Classifier
from sklearn.ensemble import GradientBoostingClassifier

model_2 = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

model_2.fit(x_train, y_train.values.ravel())

test_pred_proba = model_2.predict_proba(x_test)[:, 1]
threshold = 0.3

auc = roc_auc_score(y_test, test_pred_proba)
cm = confusion_matrix(y_test, (test_pred_proba >= threshold).astype(int))

print("AUC-ROC:", auc)
print("Confusion matrix:\n", cm)
print("\nClassification report:\n", classification_report(y_test, (test_pred_proba >= threshold).astype(int)))

AUC-ROC: 0.8627635228631374
Confusion matrix:
 [[27140   904]
 [ 1191   765]]

Classification report:
               precision    recall  f1-score   support

           0       0.96      0.97      0.96     28044
           1       0.46      0.39      0.42      1956

    accuracy                           0.93     30000
   macro avg       0.71      0.68      0.69     30000
weighted avg       0.93      0.93      0.93     30000



## Random Forest 
### Preprocessing and modeling

In [ ]:
import numpy as np 
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import RandomizedSearchCV
from xgboost.sklearn import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

params = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 5, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'class_weight': [None, 'balanced', 'balanced_subsample'],
    
}

model_3 = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    criterion='gini',
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced_subsample',
    random_state=42,

)

random_search = RandomizedSearchCV(
    estimator=model_3,
    param_distributions=params,
    n_iter=10,
    scoring='roc_auc',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

data_train = pd.read_csv('cs-training.csv')

df = data_train.copy()

def remove_na(df):
    df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
    df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())
    return df

df = remove_na(df)

late_cols = ['NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfTimes90DaysLate', 'NumberOfTime60-89DaysPastDueNotWorse']

for col in late_cols:
    df[col] = df[col].replace([96, 98], np.nan)
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    
# print(df['NumberOfTime30-59DaysPastDueNotWorse'].value_counts(),
# df['NumberOfTimes90DaysLate'].value_counts(),
# df['NumberOfTime60-89DaysPastDueNotWorse'].value_counts())

def worse_deliq(row):
    if row['NumberOfTimes90DaysLate'] > 0:
        return 3
    elif row['NumberOfTime30-59DaysPastDueNotWorse'] > 0:
        return 2
    elif row['NumberOfTime60-89DaysPastDueNotWorse'] > 0:
        return 1
    else:
        return 0
    
df['worsedeliq'] = df.apply(worse_deliq, axis=1)
df['total_time_late'] = df[late_cols].sum(axis=1)
df['total_loans'] = df['NumberOfOpenCreditLinesAndLoans'] + df['NumberRealEstateLoansOrLines']

df['debtratio_extreme_flag'] = (df['DebtRatio'] > 10).astype(int)

x_train, x_test, y_train, y_test = train_test_split(df.drop(columns=['SeriousDlqin2yrs', 'Unnamed: 0']), df['SeriousDlqin2yrs'], test_size=0.2, random_state=42)

# model_3.fit(x_train, y_train)
# test_prod_rf = model_3.predict_proba(x_test)[:, 1]

# random_search.fit(x_train, y_train)
# print("Best Parameters:", random_search.best_params_)

# best_rf = random_search.best_estimator_
# print("Best Estimator:", best_rf)
# test_prob_rf = random_search.predict_proba(x_test)[:, 1]


xgb.fit(x_train, y_train)
test_prob_xgb = xgb.predict_proba(x_test)[:, 1]

In [6]:
threshold = 0.2
# auc = roc_auc_score(y_test, test_prob_rf)
# cm = confusion_matrix(y_test, (test_prob_rf >= threshold).astype(int))

auc = roc_auc_score(y_test, test_prob_xgb)
cm = confusion_matrix(y_test, (test_prob_xgb >= threshold).astype(int))

print("AUC-ROC:", auc)
print("Confusion matrix:\n", cm)
print("Classification report: \n", classification_report(y_test, (test_prob_xgb >= threshold).astype(int)))

AUC-ROC: 0.8637538323505073
Confusion matrix:
 [[26324  1720]
 [  928  1028]]
Classification report: 
               precision    recall  f1-score   support

           0       0.97      0.94      0.95     28044
           1       0.37      0.53      0.44      1956

    accuracy                           0.91     30000
   macro avg       0.67      0.73      0.69     30000
weighted avg       0.93      0.91      0.92     30000



In [7]:
import joblib
import pandas as pd

df = pd.read_csv("cs-training.csv")

# df.head()

monthly_income_median = df['MonthlyIncome'].median()
dependents_median = df['NumberOfDependents'].median()

preprocessing_constants = {
    "monthly_income_median": float(monthly_income_median),
    "dependents_median": float(dependents_median)
}


joblib.dump(xgb, 'artifacts/gb_model.joblib')
joblib.dump(preprocessing_constants, 'artifacts/preprocessing_constants.joblib')

print("Artifacts successfully exported!")

Artifacts successfully exported!
